# Cylinder Monitor — Signal Analysis Workbench
Load a WAV recording, visualize both channels, tune detection parameters, and see what the algorithm finds.

**Workflow:**
1. Drop a WAV file into `test_data/`
2. Set the filename in the Config cell
3. Run all cells top to bottom
4. Tune parameters in the Tuning cell and re-run from there

In [ ]:
import numpy as np
import scipy.io.wavfile as wav
import scipy.signal as signal
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

plt.rcParams['figure.facecolor'] = '#1a1a1a'
plt.rcParams['axes.facecolor']   = '#111111'
plt.rcParams['axes.edgecolor']   = '#444444'
plt.rcParams['axes.labelcolor']  = '#cccccc'
plt.rcParams['xtick.color']      = '#888888'
plt.rcParams['ytick.color']      = '#888888'
plt.rcParams['text.color']       = '#cccccc'
plt.rcParams['grid.color']       = '#2a2a2a'
plt.rcParams['grid.linestyle']   = '-'
plt.rcParams['figure.figsize']   = (14, 4)

print('Packages loaded OK')

## Config — set your filename and parameters here

In [ ]:
# ── File ──────────────────────────────────────────────────────────────────────
WAV_FILE = '../test_data/recording.wav'   # <-- change this to your file

# ── Detection parameters (mirror the app Settings tab) ───────────────────────
STROKE_IN        = 1.0    # stroke length in inches
IMPACT_MULT      = 10     # threshold multiplier for T_end detection
BREAKAWAY_MULT   = 3      # threshold multiplier for T_start search
DEBOUNCE_MS      = 50     # minimum ms between two spikes
MIN_LOOKBACK_MS  = 15     # minimum ms before T_end to search for T_start
MAX_LOOKBACK_MS  = 100    # maximum ms before T_end to search for T_start
HF_BIN_LOW       = 10     # FFT bin low (~ 1 kHz at 48kHz/480 samples)
HF_BIN_HIGH      = 100    # FFT bin high (~10 kHz)
HF_FLOOR         = 0.01   # minimum HF energy to pass the FFT gate
BASELINE_PCT     = 50     # percentile of RMS history used as baseline
CHUNK_MS         = 10     # RMS chunk size in ms (must match app)

# ── Signal combination method ─────────────────────────────────────────────────
# 'additive'       |Ch0| + |Ch1|  — use for laptop/phone mic
# 'multiplicative' |Ch0| x |Ch1|  — use for wireless mics mounted on cylinder
METHOD = 'additive'

## Load WAV

In [ ]:
path = Path(WAV_FILE)
if not path.exists():
    raise FileNotFoundError(f"WAV not found: {path.resolve()}\nDrop a file into test_data/ and update WAV_FILE above.")

sr, data = wav.read(path)

# Normalise to float32 [-1, 1]
if data.dtype == np.int16:
    data = data.astype(np.float32) / 32768.0
elif data.dtype == np.int32:
    data = data.astype(np.float32) / 2147483648.0
elif data.dtype != np.float32:
    data = data.astype(np.float32)

# Handle mono
if data.ndim == 1:
    data = np.stack([data, data], axis=1)
    print('Mono file — duplicated to stereo')

ch0 = data[:, 0]
ch1 = data[:, 1]
duration_s = len(ch0) / sr
t = np.linspace(0, duration_s, len(ch0))

print(f'File:      {path.name}')
print(f'Rate:      {sr} Hz')
print(f'Duration:  {duration_s*1000:.1f} ms  ({duration_s:.2f} s)')
print(f'Samples:   {len(ch0):,}')
print(f'Ch0 peak:  {np.max(np.abs(ch0)):.4f}')
print(f'Ch1 peak:  {np.max(np.abs(ch1)):.4f}')

## Raw Waveforms — both channels

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 5), sharex=True)
fig.suptitle('Raw Waveforms', color='#4fc3f7', fontsize=13)

axes[0].plot(t * 1000, ch0, color='#4fc3f7', linewidth=0.5)
axes[0].set_ylabel('Ch0 (Left)')
axes[0].grid(True)

axes[1].plot(t * 1000, ch1, color='#81d4fa', linewidth=0.5)
axes[1].set_ylabel('Ch1 (Right)')
axes[1].set_xlabel('Time (ms)')
axes[1].grid(True)

plt.tight_layout()
plt.show()

## Combined Signal — additive vs multiplicative

In [ ]:
combined_add  = np.abs(ch0) + np.abs(ch1)
combined_mult = np.abs(ch0) * np.abs(ch1)

fig, axes = plt.subplots(2, 1, figsize=(14, 5), sharex=True)
fig.suptitle('Combined Signal', color='#4fc3f7', fontsize=13)

axes[0].plot(t * 1000, combined_add,  color='#66bb6a', linewidth=0.5)
axes[0].set_ylabel('Additive |Ch0|+|Ch1|')
axes[0].grid(True)

axes[1].plot(t * 1000, combined_mult, color='#ffa726', linewidth=0.5)
axes[1].set_ylabel('Multiplicative |Ch0|×|Ch1|')
axes[1].set_xlabel('Time (ms)')
axes[1].grid(True)

plt.tight_layout()
plt.show()

## Rolling RMS — what the app sees

In [ ]:
combined = combined_add if METHOD == 'additive' else combined_mult
chunk_samples = int(CHUNK_MS / 1000 * sr)

# Compute RMS per chunk (matches app exactly)
n_chunks = len(combined) // chunk_samples
rms_vals  = np.array([
    np.sqrt(np.mean(combined[i*chunk_samples:(i+1)*chunk_samples]**2))
    for i in range(n_chunks)
])
t_chunks  = np.array([(i + 0.5) * CHUNK_MS for i in range(n_chunks)])  # ms

baseline  = float(np.percentile(rms_vals, BASELINE_PCT))
threshold = baseline * IMPACT_MULT
bkwy_thr  = baseline * BREAKAWAY_MULT

print(f'Method:    {METHOD}')
print(f'Chunks:    {n_chunks}  ({CHUNK_MS}ms each)')
print(f'Baseline:  {baseline:.6f}  ({BASELINE_PCT}th percentile)')
print(f'Threshold: {threshold:.6f}  ({IMPACT_MULT}× baseline)')
print(f'Breakaway: {bkwy_thr:.6f}  ({BREAKAWAY_MULT}× baseline)')

fig, ax = plt.subplots(figsize=(14, 4))
ax.set_title('Rolling RMS per chunk — what the app sees', color='#4fc3f7', fontsize=13)
ax.plot(t_chunks, rms_vals, color='#4fc3f7', linewidth=0.8, label='RMS')
ax.axhline(threshold, color='#ef5350', linewidth=1.2, linestyle='--', label=f'Impact threshold ({IMPACT_MULT}×)')
ax.axhline(bkwy_thr,  color='#ffa726', linewidth=1.0, linestyle=':', label=f'Breakaway threshold ({BREAKAWAY_MULT}×)')
ax.axhline(baseline,  color='#555555', linewidth=0.8, linestyle='-',  label=f'Baseline ({BASELINE_PCT}th pct)')
ax.set_xlabel('Time (ms)')
ax.set_ylabel('RMS')
ax.legend(loc='upper right', fontsize=8)
ax.grid(True)
plt.tight_layout()
plt.show()

## Spike Detection + Lookback

In [ ]:
def compute_hf_energy(chunk, bin_low, bin_high):
    N = len(chunk)
    spectrum = np.abs(np.fft.rfft(chunk)) / N
    hi = min(bin_high, len(spectrum) - 1)
    return float(np.sum(spectrum[bin_low:hi+1]))

spikes = []    # (chunk_index, rms, hf_energy, gated)
last_spike_chunk = -999
debounce_chunks  = DEBOUNCE_MS / CHUNK_MS

for i in range(n_chunks):
    rms = rms_vals[i]
    if rms < threshold:
        continue
    chunk_data = combined[i*chunk_samples:(i+1)*chunk_samples]
    hf = compute_hf_energy(chunk_data, HF_BIN_LOW, HF_BIN_HIGH)
    if hf < HF_FLOOR:
        spikes.append({'i': i, 'rms': rms, 'hf': hf, 'gated': True})
        continue
    if i - last_spike_chunk < debounce_chunks:
        continue
    last_spike_chunk = i
    spikes.append({'i': i, 'rms': rms, 'hf': hf, 'gated': False})

live_spikes = [s for s in spikes if not s['gated']]
gated       = [s for s in spikes if s['gated']]
print(f'Spikes above threshold: {len(live_spikes)}  |  Gated by FFT: {len(gated)}')

# ── Lookback pairing ──────────────────────────────────────────────────────────
min_chunks = MIN_LOOKBACK_MS / CHUNK_MS
max_chunks = MAX_LOOKBACK_MS / CHUNK_MS
cycles = []

for idx, tend_spike in enumerate(live_spikes):
    tend_i = tend_spike['i']
    best = None
    for prev in live_spikes[:idx]:
        gap = tend_i - prev['i']
        if gap < min_chunks or gap > max_chunks:
            continue
        if prev['rms'] < bkwy_thr:
            continue
        if best is None or prev['rms'] > best['rms']:
            best = prev
    if best is not None:
        delta_ms = (tend_i - best['i']) * CHUNK_MS
        speed    = (STROKE_IN / delta_ms) * 1000
        cycles.append({
            'tstart_i': best['i'], 'tend_i': tend_i,
            'delta_ms': delta_ms,  'speed': speed,
            'tstart_rms': best['rms'], 'tend_rms': tend_spike['rms']
        })
        print(f"  Cycle: T_start={best['i']*CHUNK_MS:.0f}ms  T_end={tend_i*CHUNK_MS:.0f}ms  "
              f"Δ={delta_ms:.1f}ms  speed={speed:.2f} in/s")
    else:
        print(f"  Unmatched spike at {tend_i*CHUNK_MS:.0f}ms (no T_start in lookback)")

## Detection Map — full picture

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.set_title('Detection Map', color='#4fc3f7', fontsize=13)
ax.plot(t_chunks, rms_vals, color='#4fc3f7', linewidth=0.8, alpha=0.8, label='RMS')
ax.axhline(threshold, color='#ef5350', linewidth=1.0, linestyle='--', label='Impact threshold')
ax.axhline(bkwy_thr,  color='#ffa726', linewidth=0.8, linestyle=':',  label='Breakaway threshold')
ax.axhline(baseline,  color='#444444', linewidth=0.8,                  label='Baseline')

for s in gated:
    ax.axvline(s['i'] * CHUNK_MS, color='#888888', linewidth=0.8, alpha=0.5)

for c in cycles:
    ts_ms = c['tstart_i'] * CHUNK_MS
    te_ms = c['tend_i']   * CHUNK_MS
    ax.axvline(ts_ms, color='#ffa726', linewidth=1.5, label='T_start')
    ax.axvline(te_ms, color='#66bb6a', linewidth=1.5, label='T_end')
    ax.annotate(
        f"Δ {c['delta_ms']:.0f}ms\n{c['speed']:.1f} in/s",
        xy=((ts_ms + te_ms) / 2, threshold * 1.05),
        ha='center', va='bottom', color='#ffffff', fontsize=8,
        bbox=dict(boxstyle='round,pad=0.3', facecolor='#2a2a2a', edgecolor='#444444')
    )
    ax.axvspan(ts_ms, te_ms, alpha=0.08, color='#66bb6a')

# Deduplicate legend
handles, labels = ax.get_legend_handles_labels()
seen = set()
unique = [(h, l) for h, l in zip(handles, labels) if not (l in seen or seen.add(l))]
ax.legend(*zip(*unique), loc='upper right', fontsize=8)

ax.set_xlabel('Time (ms)')
ax.set_ylabel('RMS')
ax.grid(True)
plt.tight_layout()
plt.show()

print(f'\nSummary: {len(cycles)} cycle(s) detected')
for c in cycles:
    print(f"  Δ={c['delta_ms']:.1f}ms  speed={c['speed']:.3f} in/s  "
          f"T_start RMS={c['tstart_rms']:.5f}  T_end RMS={c['tend_rms']:.5f}")

## FFT — frequency content of detected events

In [ ]:
if not cycles:
    print('No cycles detected — nothing to show FFTs for.')
else:
    for ci, c in enumerate(cycles):
        fig, axes = plt.subplots(1, 2, figsize=(14, 4))
        fig.suptitle(f'Cycle {ci+1} — FFT at T_start and T_end', color='#4fc3f7', fontsize=12)

        for ax, spike_i, label, color in [
            (axes[0], c['tstart_i'], 'T_start (breakaway)', '#ffa726'),
            (axes[1], c['tend_i'],   'T_end (impact)',       '#66bb6a'),
        ]:
            chunk_data = combined[spike_i*chunk_samples:(spike_i+1)*chunk_samples]
            freqs  = np.fft.rfftfreq(len(chunk_data), d=1/sr)
            mags   = np.abs(np.fft.rfft(chunk_data)) / len(chunk_data)
            ax.plot(freqs / 1000, mags, color=color, linewidth=0.8)
            ax.axvspan(HF_BIN_LOW  * sr / chunk_samples / 1000,
                       HF_BIN_HIGH * sr / chunk_samples / 1000,
                       alpha=0.1, color=color, label='HF gate window')
            ax.set_title(label, color=color)
            ax.set_xlabel('Frequency (kHz)')
            ax.set_ylabel('Magnitude')
            ax.legend(fontsize=8)
            ax.grid(True)

        plt.tight_layout()
        plt.show()

## Suggested Settings
Based on what was detected — compare these against the app's Settings tab.

In [ ]:
if not live_spikes:
    print('No spikes detected — lower IMPACT_MULT or check mounting.')
else:
    rms_values   = [s['rms'] for s in live_spikes]
    hf_values    = [s['hf']  for s in live_spikes]
    rms_min      = min(rms_values)
    rms_min_mult = rms_min / baseline
    hf_min       = min(hf_values)
    MARGIN       = 0.6

    sug_impact    = max(1.5, round(rms_min_mult * MARGIN, 1))
    sug_breakaway = max(1.0, round(sug_impact * 0.5, 1))
    sug_hf        = round(hf_min * 0.8, 4)

    print('── Suggested settings (copy to app Settings tab) ──')
    print(f'  Impact multiplier:    {sug_impact}')
    print(f'  Breakaway multiplier: {sug_breakaway}')
    print(f'  HF Floor:             {sug_hf}')
    print(f'  (Based on {MARGIN*100:.0f}% of worst-case event, baseline={baseline:.6f})')
    if cycles:
        deltas = [c['delta_ms'] for c in cycles]
        print(f'\n── Lookback window recommendation ──')
        print(f'  Measured delta range: {min(deltas):.1f} – {max(deltas):.1f} ms')
        print(f'  Suggested MIN_LOOKBACK_MS: {max(5, int(min(deltas)*0.5))}')
        print(f'  Suggested MAX_LOOKBACK_MS: {int(max(deltas)*1.5)}')